In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Logical flow

per patient:
* create directory to populate with only select OSort figs for faster inspection
* create QC_df where I will manually perform QC
* for filtered neurons, incorporate spike_times, regions, coordinates for OSort .mats.
* merge as needed
* inspect & saves pt_neurs_info_df, where rows = neurons, cols = spikes, region, coords, etc. in outputs/processed_data/20{patient}/neurs_info_df.parquet

### variables

In [2]:
suffix = ['','_max','_notch600'][2]
patient = 2605
# save = False

units_dir = f'../../outputs/unit_figs'
units_all_dir, units_waves_dir = f'{units_dir}/all/20{patient}{suffix}', f'{units_dir}/waves/20{patient}{suffix}'
units_keeps_dir,units_maybes_dir,units_questionables_dir,units_mergers_dir =\
        f'{units_dir}/keeps/20{patient}', f'{units_dir}/maybes/20{patient}', f'{units_dir}/questionables/20{patient}', f'{units_dir}/mergers/20{patient}'

for dir in [units_all_dir,units_waves_dir,units_keeps_dir,units_maybes_dir,units_questionables_dir,units_mergers_dir]:    os.makedirs(dir, exist_ok=True)
print(f'total units: {len(glob.glob(f"{units_all_dir}/*"))}')

total units: 111


### 1. set up QC, populate only-cluster and only-wave fig directories

In [3]:
# parse osort figs
for file in glob.glob(f'../../data/20{patient}/osort_mat/figs{suffix}/5/*'):

    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file): # clusters
        dest = os.path.join(units_all_dir, os.path.basename(file))
        if not os.path.exists(dest): os.system(f'cp {file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')

    if 'WAVES' in os.path.basename(file): # to consider mergers
        dest = os.path.join(units_waves_dir, os.path.basename(file))
        if not os.path.exists(dest): os.system(f'cp {file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')


File already exists at ../../outputs/unit_figs/waves/202605_notch600/A202_WAVES_6_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/all/202605_notch600/A201_CL_1278_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/all/202605_notch600/A218_CL_17_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/waves/202605_notch600/A214_WAVES_1_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/waves/202605_notch600/A195_WAVES_6_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/waves/202605_notch600/A206_WAVES_1_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/all/202605_notch600/A198_CL_651_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/all/202605_notch600/A220_CL_40_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/waves/202605_notch600/A219_WAVES_1_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/wav

### create QC_df with extra columns

In [4]:
chanIDs,unitIDs = [],[]
for file in glob.glob(f'{units_all_dir}/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:])); unitIDs.append(int(os.path.basename(file).split('_')[2]))
QC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs}).sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

QC_df['keeps'], QC_df['maybes'], QC_df['questionables'], QC_df['mergers'], QC_df['notes'] = np.nan, np.nan, np.nan, np.nan, np.nan

QC_df_path = f'../../outputs/processed_data/20{patient}/QC_pt{patient}.csv'
os.makedirs(os.path.dirname(QC_df_path), exist_ok=True)
if not os.path.exists(QC_df_path): QC_df.to_csv(QC_df_path, index=False); print(f'Saving QC_df to {QC_df_path}')
else: print(f'QC_df already exists at {QC_df_path}, skipping save')
QC_df

QC_df already exists at ../../outputs/processed_data/202605/QC_pt2605.csv, skipping save


,chanID,unitID,keeps,maybes,questionables,mergers,notes
0,193,119,NaN,NaN,NaN,NaN,NaN
1,193,133,NaN,NaN,NaN,NaN,NaN
2,194,217,NaN,NaN,NaN,NaN,NaN
3,194,271,NaN,NaN,NaN,NaN,NaN
4,195,21,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
106,223,48,NaN,NaN,NaN,NaN,NaN
107,223,49,NaN,NaN,NaN,NaN,NaN
108,224,11,NaN,NaN,NaN,NaN,NaN
109,224,15,NaN,NaN,NaN,NaN,NaN


### 2. load QC; separate keeps/maybes/questionables/mergers neur figs in the data/units dir

In [5]:
QC_df = pd.read_csv(f'../../outputs/processed_data/20{patient}/QC_pt{patient}.csv')
keeps_df = QC_df[QC_df['keeps'] == 1].copy().reset_index(drop=True)
maybes_df = QC_df[~QC_df['maybes'].isna()].copy().reset_index(drop=True)
questionables_df = QC_df[QC_df['questionables'] == 1].copy().reset_index(drop=True)
mergers_df = QC_df[~QC_df['mergers'].isna()].copy().reset_index(drop=True)
# keeps + mergers
possible_neurs_info_df = pd.concat([keeps_df, mergers_df]).drop_duplicates().reset_index(drop=True)

for unit_file in glob.glob(f'{units_all_dir}/*'):
    chanID, unitID = int(os.path.basename(unit_file).split('_')[0][1:]), int(os.path.basename(unit_file).split('_')[2])

    if ((keeps_df['chanID'] == chanID) & (keeps_df['unitID'] == unitID)).any():
        dest = f'{units_keeps_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')
    if ((maybes_df['chanID'] == chanID) & (maybes_df['unitID'] == unitID)).any():
        dest = f'{units_maybes_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')
    if ((questionables_df['chanID'] == chanID) & (questionables_df['unitID'] == unitID)).any():
        dest = f'{units_questionables_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')
    if ((mergers_df['chanID'] == chanID) & (mergers_df['unitID'] == unitID)).any():
        dest = f'{units_mergers_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')

# checks
for label, df, target_dir in [('keeps', keeps_df, units_keeps_dir), ('maybes', maybes_df, units_maybes_dir), ('mergers', mergers_df, units_mergers_dir)]:
    dir_files, df_files = set(os.path.basename(f) for f in glob.glob(f'{target_dir}/*.png')), set()
    for _, row in df.iterrows():
        matches = glob.glob(f'{units_all_dir}/A{int(row["chanID"])}_*_{int(row["unitID"])}_*.png')
        df_files.update(os.path.basename(f) for f in matches)
    extra_in_dir, missing_from_dir = dir_files-df_files, df_files-dir_files
    if extra_in_dir:   print(f'[{label}] extra files (in dir, not in df): {extra_in_dir}')
    if missing_from_dir: print(f'[{label}] missing files (in df, not in dir): {missing_from_dir}')

assert len(keeps_df) == len(glob.glob(f'{units_keeps_dir}/*.png')), f'Length mismatch: keeps_df has {len(keeps_df)} rows, but {len(glob.glob(f"{units_keeps_dir}/*.png"))} files in {units_keeps_dir}'
assert len(maybes_df) == len(glob.glob(f'{units_maybes_dir}/*.png')), f'Length mismatch: maybes_df has {len(maybes_df)} rows, but {len(glob.glob(f"{units_maybes_dir}/*.png"))} files in {units_maybes_dir}'
# assert len(mergers_df) == len(glob.glob(f'{units_mergers_dir}/*.png')), f'Length mismatch: mergers_df has {len(mergers_df)} rows, but {len(glob.glob(f"{units_mergers_dir}/*.png"))} files in {units_mergers_dir}'

possible_neurs_info_df

File already exists at ../../outputs/unit_figs/keeps/202605/A202_CL_4980_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202605/A199_CL_578_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202605/A202_CL_4503_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202605/A195_CL_1447_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/maybes/202605/A202_CL_4696_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202605/A202_CL_4973_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202605/A201_CL_1349_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/maybes/202605/A205_CL_573_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202605/A202_CL_5010_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/maybes/202605/A206_CL_1641_THM_1.png, skipping copy


,chanID,unitID,keeps,maybes,questionables,mergers,notes
0,195,1447,1.0,NaN,NaN,NaN,NaN
1,199,578,1.0,NaN,NaN,NaN,NaN
2,201,1349,1.0,NaN,NaN,NaN,NaN
3,202,4503,1.0,NaN,NaN,NaN,NaN
4,202,4973,1.0,NaN,NaN,NaN,NaN
5,202,4980,1.0,NaN,NaN,NaN,NaN
6,202,5010,1.0,NaN,NaN,NaN,NaN


### 3. create neurs_info_df for neurs in possible_neurs_info_df and taking in spike times from osort mat files

### helpers

In [6]:
def getunitID2spikes(unitIDs, spikes, possible_neurs_info_df):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''    
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):
        if unitID not in possible_neurs_info_df['unitID'].tolist(): continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize
        unit2spikes[unitID].append(spike)

    return unit2spikes

### First, add spike data from OSort mats.

In [7]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../data/20{patient}/osort_mat/sorted_mats{suffix}/5/*_sorted_new.mat'):

    chanMat, chanID = sio.loadmat(mat_file), int(os.path.basename(mat_file).split('_')[0][1:])

    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # #n_spikes
    unit_vector = chanMat['assignedNegative'][0] # #n_units

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector, possible_neurs_info_df)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID); unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes)); FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

pt_neurs_info_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
pt_neurs_info_df = pd.merge(pt_neurs_info_df, possible_neurs_info_df[['chanID', 'unitID', 'keeps', 'mergers']], on=['chanID', 'unitID'], how='left')
pt_neurs_info_df = pt_neurs_info_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
assert len(pt_neurs_info_df) == len(possible_neurs_info_df), f'Length mismatch: pt_neurs_info_df has {len(pt_neurs_info_df)} rows, possible_neurs_info_df has {len(possible_neurs_info_df)} rows'
pt_neurs_info_df


,chanID,unitID,spikes,num_spikes,FR,keeps,mergers
0,195,1447,"[0.5549, 0.8193333333333334, 1.135, 2.02086666...",3619,1.875813,1.0,NaN
1,199,578,"[0.8219333333333334, 1.2233666666666667, 2.802...",1471,0.762801,1.0,NaN
2,201,1349,"[1.4040666666666668, 1.5776666666666668, 3.703...",3873,2.006766,1.0,NaN
3,202,4503,"[7.129266666666667, 12.6179, 12.80873333333333...",2146,1.114465,1.0,NaN
4,202,4973,"[5.458066666666667, 8.2871, 8.315700000000001,...",873,0.453420,1.0,NaN
5,202,4980,"[0.30323333333333335, 0.3473, 0.55403333333333...",9458,4.894157,1.0,NaN
6,202,5010,"[0.4261666666666667, 0.8591666666666667, 1.400...",10142,5.248896,1.0,NaN


### add region column by mapping channel -> region (label)

In [8]:
chanMap = sio.loadmat(glob.glob(f'../../data/20{patient}/records/*ChannelMap*.mat')[0])
labelMap = chanMap['LabelMap'].flatten(); labelMap = np.array([str(label.squeeze()) for label in labelMap])

# patient 21's spike data skips a bank of 8 channels above 208; shift back to match the channel map
if patient == 2521: pt_neurs_info_df['chanID'] = np.where(pt_neurs_info_df['chanID'] >= 209, pt_neurs_info_df['chanID'] - 8, pt_neurs_info_df['chanID'])

# which of ChannelMap1/ChannelMap2 is correct varies by patient: the correct one maps all our spike channels without nans
for key in ['ChannelMap1', 'ChannelMap2']:
    channelMap = chanMap[key].flatten()
    channel2label = dict(zip(channelMap[~np.isnan(channelMap)], labelMap[~np.isnan(channelMap)]))
    if pt_neurs_info_df['chanID'].isin(channel2label).all(): print(f'using {key}'); break
pt_neurs_info_df['region'] = pt_neurs_info_df['chanID'].map(channel2label)
assert pt_neurs_info_df['region'].notna().all(), f'unmapped channels: {pt_neurs_info_df.loc[pt_neurs_info_df["region"].isna(), "chanID"].tolist()}'

pt_neurs_info_df

using ChannelMap1


,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region
0,195,1447,"[0.5549, 0.8193333333333334, 1.135, 2.02086666...",3619,1.875813,1.0,NaN,mRDACC3
1,199,578,"[0.8219333333333334, 1.2233666666666667, 2.802...",1471,0.762801,1.0,NaN,mRDACC7
2,201,1349,"[1.4040666666666668, 1.5776666666666668, 3.703...",3873,2.006766,1.0,NaN,mLAHIP1
3,202,4503,"[7.129266666666667, 12.6179, 12.80873333333333...",2146,1.114465,1.0,NaN,mLAHIP2
4,202,4973,"[5.458066666666667, 8.2871, 8.315700000000001,...",873,0.453420,1.0,NaN,mLAHIP2
5,202,4980,"[0.30323333333333335, 0.3473, 0.55403333333333...",9458,4.894157,1.0,NaN,mLAHIP2
6,202,5010,"[0.4261666666666667, 0.8591666666666667, 1.400...",10142,5.248896,1.0,NaN,mLAHIP2


### add coordinates columns by mapping region->coords

In [9]:
def clean_entry(x): # cleaning function to handle nested arrays and bytes
    while isinstance(x, (np.ndarray, list)):    x = x[0]
    if isinstance(x, (bytes, bytearray)):    x = x.decode("utf-8", errors="ignore")
    return str(x)

In [10]:
electrodeInfo = sio.loadmat(glob.glob(f'../../data/20{patient}/records/*Electrode*.mat')[0])

ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']); region_s = ElecMapRaw[0].apply(clean_entry)
region2id_df = pd.DataFrame({"region":region_s.values, "ID":np.arange(len(region_s))})

ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})

# ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?
# atlas_index = 0; atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),"atlas_region": atlas_s.values})

pt_neurs_info_df = (pt_neurs_info_df.merge(region2id_df, on='region', how='left').merge(id2xyz_df, on='ID', how='left')) # .merge(xyz2atlasRegions, on='ID', how='left')
pt_neurs_info_df = pt_neurs_info_df.drop(columns=['ID']); pt_neurs_info_df = pt_neurs_info_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
pt_neurs_info_df

,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region,x,y,z
0,195,1447,"[0.5549, 0.8193333333333334, 1.135, 2.02086666...",3619,1.875813,1.0,NaN,mRDACC3,5.800004,35.399993,21.999995
1,199,578,"[0.8219333333333334, 1.2233666666666667, 2.802...",1471,0.762801,1.0,NaN,mRDACC7,8.200004,35.399993,21.999995
2,201,1349,"[1.4040666666666668, 1.5776666666666668, 3.703...",3873,2.006766,1.0,NaN,mLAHIP1,-15.399995,-0.200005,-15.600003
3,202,4503,"[7.129266666666667, 12.6179, 12.80873333333333...",2146,1.114465,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003
4,202,4973,"[5.458066666666667, 8.2871, 8.315700000000001,...",873,0.453420,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003
5,202,4980,"[0.30323333333333335, 0.3473, 0.55403333333333...",9458,4.894157,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003
6,202,5010,"[0.4261666666666667, 0.8591666666666667, 1.400...",10142,5.248896,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003


### 4. Merge clusters as needed

In [11]:
merged_rows = []; merge_mask = pt_neurs_info_df['mergers'].notna()

# extract merger rows as df
for _, merge_group in pt_neurs_info_df[merge_mask].groupby('mergers'):
    merged_spikes = np.sort(np.concatenate(merge_group['spikes'].values))

    first_row = merge_group.iloc[0] # populates chanID, region, merge_cluster, x, y, z
    merged_rows.append({
        **{col: first_row[col] for col in pt_neurs_info_df.columns},
        'unitID':'/'.join(merge_group['unitID'].astype(str).tolist()), 'spikes':merged_spikes,
        'num_spikes': len(merged_spikes), 'FR': len(merged_spikes) / (merged_spikes[-1] - merged_spikes[0]),
        'keeps':      1,
    })

# concat original df without merged rows + new merged rows
pt_neurs_info_df = pd.concat([pt_neurs_info_df[~merge_mask], pd.DataFrame(merged_rows)], ignore_index=True); pt_neurs_info_df = pt_neurs_info_df.sort_values(by=['chanID']).reset_index(drop=True)
pt_neurs_info_df

,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region,x,y,z
0,195,1447,"[0.5549, 0.8193333333333334, 1.135, 2.02086666...",3619,1.875813,1.0,NaN,mRDACC3,5.800004,35.399993,21.999995
1,199,578,"[0.8219333333333334, 1.2233666666666667, 2.802...",1471,0.762801,1.0,NaN,mRDACC7,8.200004,35.399993,21.999995
2,201,1349,"[1.4040666666666668, 1.5776666666666668, 3.703...",3873,2.006766,1.0,NaN,mLAHIP1,-15.399995,-0.200005,-15.600003
3,202,4503,"[7.129266666666667, 12.6179, 12.80873333333333...",2146,1.114465,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003
4,202,4973,"[5.458066666666667, 8.2871, 8.315700000000001,...",873,0.453420,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003
5,202,4980,"[0.30323333333333335, 0.3473, 0.55403333333333...",9458,4.894157,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003
6,202,5010,"[0.4261666666666667, 0.8591666666666667, 1.400...",10142,5.248896,1.0,NaN,mLAHIP2,-15.399995,-2.600005,-15.600003


### 5. inspect, typecast, & save

In [12]:
pt_neurs_info_df['spikes'] = pt_neurs_info_df['spikes'].apply(lambda x: np.array(x))
print('example neuron')
print(f'last 5 spikes (s): {pt_neurs_info_df["spikes"].iloc[0][-5:]}\nlast 5 spikes (min): {pt_neurs_info_df["spikes"].iloc[0][-5:]/60}')
pt_neurs_info_df['unitID'] = pt_neurs_info_df['unitID'].astype(str)
if 'patient' not in pt_neurs_info_df: pt_neurs_info_df.insert(0, 'patient', patient)

processed_data_dir = f'../../outputs/processed_data/20{patient}'; os.makedirs(processed_data_dir, exist_ok=True)
parquet_path = os.path.join(processed_data_dir, 'neurs_info_df.parquet')
if not os.path.exists(parquet_path):    pt_neurs_info_df.to_parquet(parquet_path, index=False); print('saving pt_neurs_info_df')
else: print(f'pt_neurs_info_df already exists at {parquet_path}, skipping save')    


example neuron
last 5 spikes (s): [1928.1567     1928.65436667 1928.8763     1929.7414     1929.85196667]
last 5 spikes (min): [32.135945   32.14423944 32.14793833 32.16235667 32.16419944]
pt_neurs_info_df already exists at ../../outputs/processed_data/202605/neurs_info_df.parquet, skipping save
